In [ ]:
# 🔍 QUICK CONNECTIVITY TEST (Run in separate cell)
import socket, logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

HOSTS = {
    "spark-master": ("spark-master", 7077),
    "nessie": ("host.docker.internal", 19120),
    "minio": ("host.docker.internal", 9000),
    "postgres": ("host.docker.internal", 5432)
}

for name, (host, port) in HOSTS.items():
    try:
        sock = socket.create_connection((host, port), timeout=5)
        sock.close()
        logger.info(f"✅ {name:12} @ {host}:{port}")
    except Exception as e:
        logger.error(f"❌ {name:12} @ {host}:{port} — {e}")

In [ ]:
# check the jars
import os
jar_path = "/jars"
print(f"📁 Contents of {jar_path}:")
for f in sorted(os.listdir(jar_path)):
    print(f"   - {f}")

In [3]:
# 🚀 ETL: PostgreSQL → Iceberg (Nessie) → MinIO [FINAL + S3 FIX]
# ✅ Pre-creates MinIO bucket check + better S3 error handling
# =============================================================================

import os, sys, time, logging, socket
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, to_date, lit

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', stream=sys.stdout)
logger = logging.getLogger(__name__)
logger.info(f"🚀 Lakehouse ETL started at {datetime.now()}")

# Config
cfg = {
    "postgres": {
        "host": os.getenv("POSTGRES_HOST", "host.docker.internal"),
        "port": os.getenv("POSTGRES_PORT", "5432"),
        "database": os.getenv("POSTGRES_DB", "learning_db"),
        "user": os.getenv("POSTGRES_USER", "learner"),
        "password": os.getenv("POSTGRES_PASSWORD", "secret"),
        "table": "public.products2iceberg",
        "driver": "org.postgresql.Driver",
    },
    "minio": {
        "endpoint": os.getenv("MINIO_ENDPOINT", "host.docker.internal:9000"),
        "access_key": "admin",
        "secret_key": "Admin@MinIO123",
        "bucket": "iceberg-warehouse",
    },
    "nessie": {
        "uri": "http://host.docker.internal:19120/api/v2",
        "ref": "main",
        "warehouse": f"s3a://{os.getenv('MINIO_BUCKET', 'iceberg-warehouse')}",
    }
}

# =============================================================================
# 🔹 Step 1: Spark Session
# =============================================================================
logger.info("🚀 Initializing Spark Session...")

spark = SparkSession.builder \
    .appName("postgres_to_iceberg_poc") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", "jupyter-lab") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.sql.extensions", 
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions," +
            "org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.type", "nessie") \
    .config("spark.sql.catalog.nessie.uri", cfg["nessie"]["uri"]) \
    .config("spark.sql.catalog.nessie.ref", cfg["nessie"]["ref"]) \
    .config("spark.sql.catalog.nessie.warehouse", cfg["nessie"]["warehouse"]) \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", cfg["minio"]["access_key"]) \
    .config("spark.hadoop.fs.s3a.secret.key", cfg["minio"]["secret_key"]) \
    .config("spark.hadoop.fs.s3a.endpoint", f"http://{cfg['minio']['endpoint']}") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", 
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.socket.recv.buffer", "65536") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.executor.heartbeatInterval", "30s") \
    .config("spark.network.timeout", "300s") \
    .getOrCreate()

logger.info(f"✅ Spark {spark.version} @ {spark.sparkContext.master}")

# =============================================================================
# 🔹 Pre-Flight: Verify S3/MinIO Access
# =============================================================================
logger.info("\n🔍 Pre-flight: Checking MinIO/S3 access...")
try:
    # Test S3A filesystem from driver (workers will use same config)
    fs = spark.sparkContext._jsc.hadoopFileSystem()
    s3_path = f"s3a://{cfg['minio']['bucket']}/"
    
    # Try to list contents (creates path if missing)
    status = fs.listStatus(spark._jvm.org.apache.hadoop.fs.Path(s3_path))
    logger.info(f"✅ S3A access OK: {len(status)} items in {s3_path}")
    
except Exception as e:
    logger.error(f"❌ S3A access failed: {e}")
    logger.error("💡 Troubleshooting:")
    logger.error("   1. Create MinIO bucket: docker exec minio mc mb local/iceberg-warehouse")
    logger.error("   2. Test worker→MinIO: docker exec spark-worker nc -zv host.docker.internal 9000")
    logger.error("   3. Verify credentials: admin / Admin@MinIO123")
    # Continue anyway - Iceberg might still work with different error handling

# =============================================================================
# 🔹 Step 2: EXTRACT - PostgreSQL
# =============================================================================
logger.info("\n📥 EXTRACT: Reading from PostgreSQL...")
extract_start = time.time()

pg_df = spark.read.format("jdbc") \
    .option("url", f"jdbc:postgresql://{cfg['postgres']['host']}:{cfg['postgres']['port']}/{cfg['postgres']['database']}") \
    .option("dbtable", cfg["postgres"]["table"]) \
    .option("user", cfg["postgres"]["user"]) \
    .option("password", cfg["postgres"]["password"]) \
    .option("driver", cfg["postgres"]["driver"]) \
    .option("numPartitions", "1") \
    .option("fetchsize", "1000") \
    .option("connectTimeout", "10") \
    .option("socketTimeout", "30") \
    .load()

row_count = pg_df.count()
logger.info(f"✅ Extracted {row_count} rows in {time.time()-extract_start:.1f}s")
pg_df.show(3, truncate=False)

# =============================================================================
# 🔹 Step 3: TRANSFORM
# =============================================================================
logger.info("\n🔄 TRANSFORM: Adding metadata...")
transform_start = time.time()

batch_id = f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
iceberg_df = pg_df \
    .withColumn("ingested_at", current_timestamp()) \
    .withColumn("ingested_date", to_date(col("ingested_at"))) \
    .withColumn("source_system", lit("postgres_source")) \
    .withColumn("batch_id", lit(batch_id))

if "status" in pg_df.columns:
    iceberg_df = iceberg_df.filter(col("status") != "deleted")
if "price" in pg_df.columns:
    iceberg_df = iceberg_df.withColumn("price_usd", col("price").cast("decimal(12,2)"))

result_count = iceberg_df.count()
logger.info(f"✅ Transformed: {row_count} → {result_count} rows in {time.time()-transform_start:.1f}s")

# =============================================================================
# 🔹 Step 4: LOAD - Iceberg via Nessie [FIXED: S3 pre-check + fallback]
# =============================================================================
logger.info("\n📤 LOAD: Writing to Iceberg...")
load_start = time.time()
TABLE_NAME = "nessie.cold_data.products2iceberg"

try:
    # Create namespace
    spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.cold_data")
    logger.info("✅ Namespace 'nessie.cold_data' ready")
    
    # Check if table exists
    table_exists = False
    try:
        existing = spark.sql(f"SHOW TABLES IN nessie.cold_data LIKE 'products2iceberg'").collect()
        table_exists = len(existing) > 0
    except:
        pass
    logger.info(f"🔍 Table exists: {table_exists}")
    
    # Build writer
    writer = (iceberg_df.writeTo(TABLE_NAME)
        .using("iceberg")
        .tableProperty("format", "parquet")
        .tableProperty("compression", "snappy")
        .tableProperty("write.object-enabled", "true")  # Helps with S3 small files
    )
    
    if result_count >= 1000:
        writer = writer.partitionedBy("ingested_date")
    
    # Execute write with retry logic
    max_retries = 2
    for attempt in range(max_retries + 1):
        try:
            if table_exists:
                writer.append()
            else:
                writer.createOrReplace()
            logger.info(f"✅ Write succeeded on attempt {attempt+1}")
            break
        except Exception as write_err:
            if "Failed to create Parquet file" in str(write_err) and attempt < max_retries:
                logger.warning(f"⚠️ S3 write failed (attempt {attempt+1}/{max_retries+1}), retrying...")
                time.sleep(5)  # Brief pause before retry
                continue
            else:
                raise
    
    load_time = time.time() - load_start
    logger.info(f"✅ Wrote {result_count} rows to Iceberg in {load_time:.1f}s")
    
except Exception as e:
    logger.error(f"❌ Iceberg write failed: {e}")
    
    # 🔹 Fallback: SQL API with explicit S3 path
    logger.warning("🔄 Falling back to SQL API with explicit S3 location...")
    try:
        iceberg_df.createOrReplaceTempView("temp_data")
        
        # Create table with explicit S3 location (bypasses Nessie catalog if needed)
        s3_location = f"s3a://{cfg['minio']['bucket']}/cold_data/products2iceberg"
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
                id INT, name STRING, description STRING, price DECIMAL(10,2),
                created_at TIMESTAMP, ingested_at TIMESTAMP, ingested_date DATE,
                source_system STRING, batch_id STRING, price_usd DECIMAL(12,2)
            ) USING iceberg
            LOCATION '{s3_location}'
            TBLPROPERTIES ('format'='parquet', 'compression'='snappy')
        """)
        
        spark.sql(f"INSERT INTO {TABLE_NAME} SELECT * FROM temp_data")
        logger.info("✅ SQL API fallback succeeded")
        
    except Exception as e2:
        logger.error(f"❌ Fallback also failed: {e2}")
        logger.error("💡 Final troubleshooting:")
        logger.error("   1. Create MinIO bucket: docker exec minio mc mb local/iceberg-warehouse")
        logger.error("   2. Verify worker can reach MinIO: docker exec spark-worker curl -s http://host.docker.internal:9000/minio/health/live")
        logger.error("   3. Check MinIO logs: docker logs minio | grep -i error")
        raise

# =============================================================================
# 🔹 Step 5: VERIFY
# =============================================================================
logger.info("\n🔍 VERIFY: Reading from Iceberg...")
verify_df = spark.sql(f"SELECT * FROM {TABLE_NAME} LIMIT 5")
verify_df.show(truncate=False)

stats = spark.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(ingested_date) as earliest, MAX(ingested_date) as latest
    FROM {TABLE_NAME}
""")
logger.info("📊 Table statistics:")
stats.show(truncate=False)

# =============================================================================
# 🔹 Summary
# =============================================================================
total_time = time.time() - extract_start
logger.info("\n" + "🎉"*30)
logger.info("✨ ETL COMPLETE ✨")
logger.info("🎉"*30)
logger.info(f"⏱️  Total: {total_time:.1f}s | Rows: {result_count}")
logger.info(f"🗄️  Table: {TABLE_NAME}")
logger.info(f"🔗 Dremio: http://host.docker.internal:9047 → nessie → cold_data")
logger.info(f"🔗 MinIO:  http://host.docker.internal:9001 → iceberg-warehouse/")
logger.info(f"🔗 Nessie: http://host.docker.internal:19120 → Time-travel history")
logger.info("🎉"*30)

spark.stop()
logger.info("✅ Done!")

2026-03-31 09:56:01,756 [INFO] 🚀 Lakehouse ETL started at 2026-03-31 09:56:01.756308
2026-03-31 09:56:01,778 [INFO] 🚀 Initializing Spark Session...
2026-03-31 09:56:06,919 [INFO] ✅ Spark 3.5.0 @ spark://spark-master:7077
2026-03-31 09:56:06,920 [INFO] 
🔍 Pre-flight: Checking MinIO/S3 access...
2026-03-31 09:56:06,929 [ERROR] ❌ S3A access failed: An error occurred while calling o278.hadoopFileSystem. Trace:
py4j.Py4JException: Method hadoopFileSystem([]) does not exist
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:321)
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:329)
	at py4j.Gateway.invoke(Gateway.java:274)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:106)
	at java.base/java.lang.Thread.run(Thread.java:840)


2

In [4]:
# 🚀 ETL: PostgreSQL → Iceberg (Nessie) → MinIO [FINAL + new table
# ✅ Pre-creates MinIO bucket check + better S3 error handling
# =============================================================================

import os, sys, time, logging, socket
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, to_date, lit

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', stream=sys.stdout)
logger = logging.getLogger(__name__)
logger.info(f"🚀 Lakehouse ETL started at {datetime.now()}")

# Config
cfg = {
    "postgres": {
        "host": os.getenv("POSTGRES_HOST", "host.docker.internal"),
        "port": os.getenv("POSTGRES_PORT", "5432"),
        "database": os.getenv("POSTGRES_DB", "learning_db"),
        "user": os.getenv("POSTGRES_USER", "learner"),
        "password": os.getenv("POSTGRES_PASSWORD", "secret"),
        "table": "public.products2iceberg",
        "driver": "org.postgresql.Driver",
    },
    "minio": {
        "endpoint": os.getenv("MINIO_ENDPOINT", "host.docker.internal:9000"),
        "access_key": "admin",
        "secret_key": "Admin@MinIO123",
        "bucket": "iceberg-warehouse",
    },
    "nessie": {
        "uri": "http://host.docker.internal:19120/api/v2",
        "ref": "main",
        "warehouse": f"s3a://{os.getenv('MINIO_BUCKET', 'iceberg-warehouse')}",
    }
}

# =============================================================================
# 🔹 Step 1: Spark Session
# =============================================================================
logger.info("🚀 Initializing Spark Session...")

spark = SparkSession.builder \
    .appName("postgres_to_iceberg_poc") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", "jupyter-lab") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.sql.extensions", 
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions," +
            "org.projectnessie.spark.extensions.NessieSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.type", "nessie") \
    .config("spark.sql.catalog.nessie.uri", cfg["nessie"]["uri"]) \
    .config("spark.sql.catalog.nessie.ref", cfg["nessie"]["ref"]) \
    .config("spark.sql.catalog.nessie.warehouse", cfg["nessie"]["warehouse"]) \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", cfg["minio"]["access_key"]) \
    .config("spark.hadoop.fs.s3a.secret.key", cfg["minio"]["secret_key"]) \
    .config("spark.hadoop.fs.s3a.endpoint", f"http://{cfg['minio']['endpoint']}") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", 
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.socket.recv.buffer", "65536") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.executor.heartbeatInterval", "30s") \
    .config("spark.network.timeout", "300s") \
    .getOrCreate()

logger.info(f"✅ Spark {spark.version} @ {spark.sparkContext.master}")

# =============================================================================
# 🔹 Pre-Flight: Verify S3/MinIO Access
# =============================================================================
logger.info("\n🔍 Pre-flight: Checking MinIO/S3 access...")
try:
    # Test S3A filesystem from driver (workers will use same config)
    fs = spark.sparkContext._jsc.hadoopFileSystem()
    s3_path = f"s3a://{cfg['minio']['bucket']}/"
    
    # Try to list contents (creates path if missing)
    status = fs.listStatus(spark._jvm.org.apache.hadoop.fs.Path(s3_path))
    logger.info(f"✅ S3A access OK: {len(status)} items in {s3_path}")
    
except Exception as e:
    logger.error(f"❌ S3A access failed: {e}")
    logger.error("💡 Troubleshooting:")
    logger.error("   1. Create MinIO bucket: docker exec minio mc mb local/iceberg-warehouse")
    logger.error("   2. Test worker→MinIO: docker exec spark-worker nc -zv host.docker.internal 9000")
    logger.error("   3. Verify credentials: admin / Admin@MinIO123")
    # Continue anyway - Iceberg might still work with different error handling

# =============================================================================
# 🔹 Step 2: EXTRACT - PostgreSQL
# =============================================================================
logger.info("\n📥 EXTRACT: Reading from PostgreSQL...")
extract_start = time.time()

pg_df = spark.read.format("jdbc") \
    .option("url", f"jdbc:postgresql://{cfg['postgres']['host']}:{cfg['postgres']['port']}/{cfg['postgres']['database']}") \
    .option("dbtable", cfg["postgres"]["table"]) \
    .option("user", cfg["postgres"]["user"]) \
    .option("password", cfg["postgres"]["password"]) \
    .option("driver", cfg["postgres"]["driver"]) \
    .option("numPartitions", "1") \
    .option("fetchsize", "1000") \
    .option("connectTimeout", "10") \
    .option("socketTimeout", "30") \
    .load()

row_count = pg_df.count()
logger.info(f"✅ Extracted {row_count} rows in {time.time()-extract_start:.1f}s")
pg_df.show(3, truncate=False)

# =============================================================================
# 🔹 Step 3: TRANSFORM
# =============================================================================
logger.info("\n🔄 TRANSFORM: Adding metadata...")
transform_start = time.time()

batch_id = f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
iceberg_df = pg_df \
    .withColumn("ingested_at", current_timestamp()) \
    .withColumn("ingested_date", to_date(col("ingested_at"))) \
    .withColumn("source_system", lit("postgres_source")) \
    .withColumn("batch_id", lit(batch_id))

if "status" in pg_df.columns:
    iceberg_df = iceberg_df.filter(col("status") != "deleted")
if "price" in pg_df.columns:
    iceberg_df = iceberg_df.withColumn("price_usd", col("price").cast("decimal(12,2)"))

result_count = iceberg_df.count()
logger.info(f"✅ Transformed: {row_count} → {result_count} rows in {time.time()-transform_start:.1f}s")

# =============================================================================
# 🔹 Step 4: LOAD - Iceberg via Nessie [FIXED: S3 pre-check + fallback]
# =============================================================================
logger.info("\n📤 LOAD: Writing to Iceberg...")
load_start = time.time()
TABLE_NAME = "nessie.cold_data.products2icebergDemo3"

try:
    # Create namespace
    spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.cold_data")
    logger.info("✅ Namespace 'nessie.cold_data' ready")
    
    # Check if table exists
    table_exists = False
    try:
        existing = spark.sql(f"SHOW TABLES IN nessie.cold_data LIKE 'products2icebergDemo3'").collect()
        table_exists = len(existing) > 0
    except:
        pass
    logger.info(f"🔍 Table exists: {table_exists}")
    
    # Build writer
    writer = (iceberg_df.writeTo(TABLE_NAME)
        .using("iceberg")
        .tableProperty("format", "parquet")
        .tableProperty("compression", "snappy")
        .tableProperty("write.object-enabled", "true")  # Helps with S3 small files
    )
    
    if result_count >= 1000:
        writer = writer.partitionedBy("ingested_date")
    
    # Execute write with retry logic
    max_retries = 2
    for attempt in range(max_retries + 1):
        try:
            if table_exists:
                writer.append()
            else:
                writer.createOrReplace()
            logger.info(f"✅ Write succeeded on attempt {attempt+1}")
            break
        except Exception as write_err:
            if "Failed to create Parquet file" in str(write_err) and attempt < max_retries:
                logger.warning(f"⚠️ S3 write failed (attempt {attempt+1}/{max_retries+1}), retrying...")
                time.sleep(5)  # Brief pause before retry
                continue
            else:
                raise
    
    load_time = time.time() - load_start
    logger.info(f"✅ Wrote {result_count} rows to Iceberg in {load_time:.1f}s")
    
except Exception as e:
    logger.error(f"❌ Iceberg write failed: {e}")
    
    # 🔹 Fallback: SQL API with explicit S3 path
    logger.warning("🔄 Falling back to SQL API with explicit S3 location...")
    try:
        iceberg_df.createOrReplaceTempView("temp_data")
        
        # Create table with explicit S3 location (bypasses Nessie catalog if needed)
        s3_location = f"s3a://{cfg['minio']['bucket']}/cold_data/products2icebergDemo3"
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
                id INT, name STRING, description STRING, price DECIMAL(10,2),
                created_at TIMESTAMP, ingested_at TIMESTAMP, ingested_date DATE,
                source_system STRING, batch_id STRING, price_usd DECIMAL(12,2)
            ) USING iceberg
            LOCATION '{s3_location}'
            TBLPROPERTIES ('format'='parquet', 'compression'='snappy')
        """)
        
        spark.sql(f"INSERT INTO {TABLE_NAME} SELECT * FROM temp_data")
        logger.info("✅ SQL API fallback succeeded")
        
    except Exception as e2:
        logger.error(f"❌ Fallback also failed: {e2}")
        logger.error("💡 Final troubleshooting:")
        logger.error("   1. Create MinIO bucket: docker exec minio mc mb local/iceberg-warehouse")
        logger.error("   2. Verify worker can reach MinIO: docker exec spark-worker curl -s http://host.docker.internal:9000/minio/health/live")
        logger.error("   3. Check MinIO logs: docker logs minio | grep -i error")
        raise

# =============================================================================
# 🔹 Step 5: VERIFY
# =============================================================================
logger.info("\n🔍 VERIFY: Reading from Iceberg...")
verify_df = spark.sql(f"SELECT * FROM {TABLE_NAME} LIMIT 5")
verify_df.show(truncate=False)

stats = spark.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(ingested_date) as earliest, MAX(ingested_date) as latest
    FROM {TABLE_NAME}
""")
logger.info("📊 Table statistics:")
stats.show(truncate=False)

# =============================================================================
# 🔹 Summary
# =============================================================================
total_time = time.time() - extract_start
logger.info("\n" + "🎉"*30)
logger.info("✨ ETL COMPLETE ✨")
logger.info("🎉"*30)
logger.info(f"⏱️  Total: {total_time:.1f}s | Rows: {result_count}")
logger.info(f"🗄️  Table: {TABLE_NAME}")
logger.info(f"🔗 Dremio: http://host.docker.internal:9047 → nessie → cold_data")
logger.info(f"🔗 MinIO:  http://host.docker.internal:9001 → iceberg-warehouse/")
logger.info(f"🔗 Nessie: http://host.docker.internal:19120 → Time-travel history")
logger.info("🎉"*30)

spark.stop()
logger.info("✅ Done!")

2026-03-31 10:03:40,291 [INFO] 🚀 Lakehouse ETL started at 2026-03-31 10:03:40.291306
2026-03-31 10:03:40,295 [INFO] 🚀 Initializing Spark Session...
2026-03-31 10:03:41,206 [INFO] ✅ Spark 3.5.0 @ spark://spark-master:7077
2026-03-31 10:03:41,208 [INFO] 
🔍 Pre-flight: Checking MinIO/S3 access...
2026-03-31 10:03:41,226 [ERROR] ❌ S3A access failed: An error occurred while calling o389.hadoopFileSystem. Trace:
py4j.Py4JException: Method hadoopFileSystem([]) does not exist
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:321)
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:329)
	at py4j.Gateway.invoke(Gateway.java:274)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:106)
	at java.base/java.lang.Thread.run(Thread.java:840)


2